# 🏦 은행 다국어 챗봇 — 통합 검증 노트북

**서식 자동완성 + 트러블슈팅 상담**을 하나의 모델로 처리하는 전체 파이프라인 검증.

**구성:**
- 모델 1개(Qwen) 공유 → 서식 추출 + 상담 RAG 둘 다 사용
- `/api/chat` 분기: scenario에 따라 서식/상담
- `/api/render`: 한국어 제출용 PDF

**사용법:** 런타임 → GPU(T4) 설정 후, 위에서 아래로 실행.

---
## 0. GPU 확인

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 1. 설치 + 코드 업로드

In [ ]:
!pip install -q transformers accelerate pymupdf langdetect
# 상담 RAG용 (ChromaDB 쓸 경우)
!pip install -q sentence-transformers chromadb
print("✅ 설치 완료")

In [ ]:
from google.colab import files
import os
if not os.path.exists("bank_form_filler") and not os.path.exists("bank_form_filler.zip"):
    print("bank_form_filler.zip 업로드:")
    files.upload()
if not os.path.exists("bank_form_filler"):
    !unzip -o -q bank_form_filler.zip
!apt-get install -y -q fonts-noto-cjk > /dev/null 2>&1
import sys; sys.path.insert(0, "/content/bank_form_filler")
DATA_DIR = "/content/bank_form_filler/app/data"
print("✅ 코드 준비 완료")

## 2. 모델 1개 로드 (서식+상담 공유)

T4면 3B, 큰 GPU면 7B. **이 모델 하나를 서식 추출과 상담이 함께 씁니다.**

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from app.services.llm_engine import LLMEngine

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"   # 큰 GPU면 "Qwen/Qwen2.5-7B-Instruct"
print(f"로딩 중: {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="auto")

# 공유 엔진 1개 생성
engine = LLMEngine(model, tokenizer)
print("✅ 모델 로드 완료 (서식+상담 공유)")

## 3. 서식 추출기 + 상담 RAG 연결 (같은 엔진)

In [ ]:
from app.services.slot_extractor_hf import HFSlotExtractor
from app.services.consult_rag import ConsultRAG, load_rag_resources

# 서식 추출기 (공유 엔진)
extractor = HFSlotExtractor(engine=engine)

# 상담 RAG (공유 엔진). ChromaDB 있으면 로드, 없으면 폴백.
# build_db로 만든 DB가 드라이브에 있으면 경로 지정:
# embedder, collection, term_dict = load_rag_resources(CHROMA_DIR, TERM_DICT_PATH)
embedder, collection, term_dict = None, None, {}  # 폴백 모드 (DB 없이 LLM 단독)
consultant = ConsultRAG(engine=engine, embedder=embedder,
                        collection=collection, term_dict=term_dict)
print("✅ 서식 추출기 + 상담 RAG 준비 완료 (모델 공유)")

## 4. 서식 시나리오 검증 (외화송금)

stateless 방식: form_state를 주고받으며 채워나갑니다.

In [ ]:
from app.services.form_handler import process_form_turn
import json

# 초기 form_state (프론트가 보내는 형태)
form_state = {"sender_name_eng": None, "remittance_amount": None,
              "beneficiary_name": None}

utterances = [
    "Tôi tên Nguyen Van An, hộ chiếu M1234567, tài khoản 110-234-567890.",
    "Tôi muốn gửi 1000 USD cho mẹ tôi tên Tran Thi Mai.",
    "Ngân hàng Vietcombank, SWIFT BFTVVNVX, Hà Nội Việt Nam. Mục đích hỗ trợ gia đình.",
]
history = []
for i, utt in enumerate(utterances, 1):
    print(f"\n{'='*55}\n👤 ({i}턴): {utt}")
    result = process_form_turn(extractor, DATA_DIR, "remittance",
                               utt, form_state, history)
    form_state = result["updated_form_state"]   # 프론트처럼 상태 누적
    history.append({"role": "user", "content": utt})
    history.append({"role": "assistant", "content": result["bot_reply"]})
    filled = {k: v for k, v in form_state.items() if v}
    print(f"📊 채워진 필드 {len(filled)}개")
    print(f"🤖 {result['bot_reply']}")
    if result["form_complete"]:
        print("✅ 서식 완성!")
        break

print("\n=== 최종 form_state ===")
print(json.dumps({k:v for k,v in form_state.items() if v}, ensure_ascii=False, indent=2))

## 5. 트러블슈팅 시나리오 검증 (상담)

서식 없이 상담 답변만. 같은 모델이 처리합니다.

In [ ]:
# 카드 분실 상담 (한국어)
print("👤 카드를 잃어버렸어요. 어떻게 해야 하나요?")
reply = consultant.answer("카드를 잃어버렸어요. 어떻게 해야 하나요?")
print(f"🤖 {reply}\n")

# 영어 상담
print("👤 How do I unblock my card?")
reply2 = consultant.answer("How do I unblock my card?")
print(f"🤖 {reply2}")

## 6. 통합 분기 검증 (/api/chat 시뮬레이션)

실제 API가 scenario로 어떻게 분기하는지 함수로 확인.

In [ ]:
def simulate_api_chat(scenario, user_message, current_form_state=None, history=None):
    """main.py의 /api/chat 로직을 그대로 시뮬레이션."""
    current_form_state = current_form_state or {}
    history = history or []
    if scenario == "troubleshooting":
        reply = consultant.answer(user_message, history)
        return {"bot_reply": reply, "updated_form_state": {}, "form_complete": False}
    else:
        return process_form_turn(extractor, DATA_DIR, scenario,
                                 user_message, current_form_state, history)

# 서식 분기
r1 = simulate_api_chat("account_opening", "Tôi tên Nguyen, sinh 1990-05-15",
                       {"customer_name": None, "birth_date": None})
print("[account_opening]", "완료" if r1["form_complete"] else "진행중")
print("  채워진:", {k:v for k,v in r1["updated_form_state"].items() if v})

# 상담 분기
r2 = simulate_api_chat("troubleshooting", "ATM에서 돈이 안 나와요")
print("\n[troubleshooting] form_state:", r2["updated_form_state"], "(빈 객체여야 정상)")
print("  답변:", r2["bot_reply"][:40])

## 7. PDF 생성 (한국어 제출용)

서식이 완성되면 실제 하나은행 PDF에 렌더링.

In [ ]:
from app.services.pdf_renderer import render_form
from app.core.slot_schema import load_form_schema
from app.core.field_mapping import front_to_backend
import fitz
from IPython.display import Image, display

# 4단계에서 채운 form_state 사용
schema = load_form_schema("foreign_remittance", DATA_DIR)
backend_slots = front_to_backend("remittance", form_state)

# 더미 서명
sd = fitz.open(); p = sd.new_page(width=120, height=40)
p.insert_text((10, 28), "signed", fontsize=16); sig = p.get_pixmap().tobytes("png"); sd.close()
sig_keys = [k for k,f in schema.fields.items() if f.field_type=="signature"]
signatures = {k: sig for k in sig_keys}

pdf_bytes = render_form(schema, backend_slots, signatures=signatures)
with open("/content/result.pdf", "wb") as f: f.write(pdf_bytes)

doc = fitz.open("/content/result.pdf")
doc[0].get_pixmap(dpi=100).save("/content/preview.png"); doc.close()
display(Image("/content/preview.png"))
print("✅ 한국어 제출용 PDF 생성 완료")

---
## 검증 체크리스트
- [ ] **2단계**: 모델 1개만 로드되는가? (GPU 메모리 확인)
- [ ] **4단계**: 서식이 stateless하게 채워지고 재질문이 사용자 언어로?
- [ ] **5단계**: 상담이 같은 모델로 답하는가?
- [ ] **6단계**: scenario로 서식/상담이 올바르게 분기되는가?
- [ ] **7단계**: PDF가 정확히 채워지는가?

**3B 한계가 보이면** 7B로 올려 재검증 (팀원과 코랩 결제 상의 후).

---
# 🚀 서버 띄우기 (프론트 연결용)

여기까지 셀을 다 실행해서 `engine`, `extractor`, `consultant`가 준비된 상태여야 합니다.
아래를 실행하면 FastAPI 서버가 켜지고, ngrok으로 공개 주소가 생깁니다.
그 주소를 프론트에게 전달하면 연결 완료.

## S1. ngrok 설치 + 인증

ngrok은 코랩(닫힌 환경)을 외부에서 접속 가능하게 뚫는 터널입니다.
무료 계정이 필요해요: https://dashboard.ngrok.com/signup → authtoken 복사

In [ ]:
!pip install -q pyngrok nest_asyncio websockets
from pyngrok import ngrok, conf

# ↓↓↓ 본인 ngrok authtoken 붙여넣기 ↓↓↓
NGROK_TOKEN = "여기에_authtoken_붙여넣기"
conf.get_default().auth_token = NGROK_TOKEN
print("✅ ngrok 준비 완료")

## S2. 서비스 연결 (init_services)

앞서 만든 추출기·상담기를 FastAPI 앱에 주입합니다.

In [ ]:
import app.main as backend
backend.init_services(extractor, consultant)
print("✅ 서비스 주입 완료")
# 상태 확인
print("  extractor:", backend._services["extractor"] is not None)
print("  consultant:", backend._services["consultant"] is not None)

## S3. 서버 실행 + 공개 주소 생성

이 셀을 실행하면 서버가 켜지고 공개 URL이 출력됩니다.
**이 셀은 계속 실행 상태로 둬야** 서버가 살아있습니다. (멈추면 서버 꺼짐)

In [ ]:
import nest_asyncio, uvicorn
from pyngrok import ngrok

nest_asyncio.apply()
PORT = 8000

# 기존 터널 정리 후 새 공개 주소 생성
ngrok.kill()
public_url = ngrok.connect(PORT).public_url
print("="*55)
print(f"🌐 프론트에게 줄 주소: {public_url}")
print("="*55)
print("\n⚠️ 이 셀은 '계속 실행 중'(로딩 표시) 상태여야 정상입니다.")
print("   체크표시(완료)가 뜨면 서버가 꺼진 것입니다.")
print("   서버를 끄려면 이 셀을 직접 '중지'하세요.\n")

# 서버를 이 셀에서 직접 실행 (셀이 여기서 계속 멈춰 있음 = 서버 살아있음)
uvicorn.run(backend.app, host="0.0.0.0", port=PORT, log_level="warning")

## S4. 서버 동작 확인 (별도로!)

⚠️ **중요:** S3 셀은 이제 "계속 실행 중" 상태로 멈춰 있습니다 (정상).
S3이 돌아가는 동안, **이 셀(S4)을 따로 실행**해서 테스트하세요.
(S3을 멈추면 안 됩니다. S3은 그대로 두고 S4만 실행)

브라우저 테스트는 `api_test.html`에 위 주소를 넣어서도 가능합니다.

In [ ]:
import requests, time
time.sleep(2)  # 서버 켜질 시간

# health 확인
r = requests.get(f"{public_url}/health")
print("health:", r.json())

# /api/chat 테스트 (외화송금)
r = requests.post(f"{public_url}/api/chat", json={
    "scenario": "remittance",
    "user_message": "Tôi tên Nguyen Van An, gửi 1000 USD",
    "current_form_state": {}
})
print("\nchat 응답:")
import json
print(json.dumps(r.json(), ensure_ascii=False, indent=2))

## S5. 프론트에게 전달할 것

1. **위 S3에서 나온 공개 주소** (`https://xxxx.ngrok.io`)
2. **`프론트_연결안내.md`** + **`FIELD_SPEC.md`**

> ⚠️ 주의:
> - ngrok 무료는 주소가 **실행할 때마다 바뀜**. 다시 띄우면 새 주소 공유.
> - 코랩은 일정 시간 후 꺼짐 → 발표 직전에 띄우고 바로 시연.
> - 서버 셀(S3)을 멈추면 서버도 꺼짐.

---
# 🏦 계좌개설 전체 실행 + PDF 생성 (체크박스·약관 포함)

실제 대화 흐름을 끝까지 돌려서 필수 정보를 다 채우고,
체크박스(직업/통신사/우편물/거래목적/자금원천/해외납세) + 약관 동의 + 서명까지
PDF에 들어가는 것을 확인합니다.

## A1. 계좌개설 대화 — 끝까지 진행

In [ ]:
from app.services.form_handler import process_form_turn
import json

# 텍스트 필수는 미리 채우고 시작 (그룹/약관 흐름 확인용)
# 실제로는 사용자가 대화로 하나씩 입력
state = {
    "customer_name": "NGUYEN VAN AN",
    "birth_date": "1995-03-15",
    "mobile_phone": "010-1234-5678",
    "address_home": "서울 영등포구 여의도동",
    "product_name": "자유적금",
    "enrollment_amount": "500000",
}

# 그룹 선택 답변 (번호)
answers = {
    "group:occupation": "1",   # 1=급여소득자 (직장항목 물어봄)
    "group:carrier": "1",      # 1=SKT
    "group:mailing": "1",      # 1=자택
    "group:purpose": "1",      # 1=급여/생활비
    "group:source": "1",       # 1=근로소득
    "group:overseas": "2",     # 2=아니오
    "consent": "동의",
}

stage = ""
msg = "계좌 만들고 싶어요"
print("=== 대화 진행 ===")
for i in range(15):
    r = process_form_turn(extractor, DATA_DIR, "account_opening",
                          msg, state, language="Korean", stage=stage)
    state = r["updated_form_state"]
    stage = r["stage"]
    print(f"{i+1}. 챗봇: {r['bot_reply'][:45].strip()}")
    if r["form_complete"]:
        print(f"\n>>> 완료! DB저장: {r.get('saved',{}).get('saved')}")
        break
    # 다음 답변 결정
    if stage in answers:
        msg = answers[stage]
        print(f"   사용자: {msg}")
    elif stage == "":  # 직장 정보 입력 차례
        msg = "회사는 서울 강남구 테헤란로 123이고 전화는 02-555-1234"
        print(f"   사용자: {msg}")

print("\n완성된 form_state의 체크 항목 수:",
      len([k for k,v in state.items() if v=="V"]))

## A2. 완성된 정보로 PDF 생성 (한국어 제출용)

In [ ]:
from app.core.slot_schema import load_form_schema
from app.core.field_mapping import front_to_backend
from app.services.pdf_renderer import render_form
import fitz

schema = load_form_schema("account_opening", DATA_DIR)
backend = front_to_backend("account_opening", state)

# 더미 서명 (실제로는 프론트가 그린 서명 PNG)
_sd = fitz.open(); _p = _sd.new_page(width=120, height=40)
_p.insert_text((10, 28), "signed", fontsize=14)
_sig = _p.get_pixmap().tobytes("png"); _sd.close()
signatures = {k: _sig for k, f in schema.fields.items()
              if f.field_type == "signature"}

pdf_bytes = render_form(schema, backend, signatures=signatures)
with open("account_opening_result.pdf", "wb") as f:
    f.write(pdf_bytes)
print(f"✅ PDF 생성 완료: account_opening_result.pdf ({len(pdf_bytes):,} bytes)")

## A3. PDF 미리보기 (체크박스 확인)

In [ ]:
import fitz
from IPython.display import Image, display

doc = fitz.open("account_opening_result.pdf")
print(f"총 {doc.page_count}페이지\n")
for i in range(doc.page_count):
    pix = doc[i].get_pixmap(dpi=110)
    pix.save(f"preview_p{i}.png")
    print(f"--- {i+1}페이지 ---")
    display(Image(f"preview_p{i}.png"))
doc.close()
print("\n확인 포인트: 통신사/우편물/직업/거래목적/자금원천/해외납세 체크 + 약관 체크 + 서명")

## A4. DB에 저장된 신청 확인

서식이 완성되면 SQLite에 필수 정보가 저장됩니다. 데모에서 '저장됨'을 보여줄 수 있어요.

In [ ]:
from app.services.db_store import list_applications, count_applications

print(f"DB에 저장된 신청: {count_applications()}건\n")
for app in list_applications():
    print(f"#{app['id']} [{app['scenario']}] {app['name']} | "
          f"{app['birth_date']} | {app['phone']}")
    print(f"   {app['extra']}")

---
# 🪪 신분증 OCR (전부 로컬 - 외부 API 미사용)

EasyOCR(로컬) + 우리 Qwen(로컬)으로 외국인등록증을 처리합니다.
**신분증이 외부로 나가지 않습니다** (Gemini 등 외부 API 미사용).
본 챗봇과 동일한 보안 원칙: 데이터가 우리 서버 밖으로 안 나감.

## O1. OCR 패키지 설치

In [ ]:
!pip install -q easyocr opencv-python-headless python-multipart
print("✅ OCR 패키지 설치 완료 (EasyOCR은 첫 실행 시 모델 다운로드)")

## O2. 신분증 OCR 테스트

신분증 이미지를 업로드해서 {name, arc_number, visa_type, nationality} 추출.
※ 실제 신분증 대신 가짜/샘플 이미지 사용 권장 (민감정보).

In [ ]:
from google.colab import files
from app.services.ocr_arc import extract_id_info

print("신분증 이미지 업로드:")
uploaded = files.upload()

for fname, data in uploaded.items():
    print(f"\n=== {fname} 처리 중 ===")
    # extractor.engine = 로컬 Qwen (외부로 안 나감)
    result = extract_id_info(data, extractor.engine)
    import json
    print(json.dumps(result, ensure_ascii=False, indent=2))
    break

## O3. (참고) API로 호출하는 법

서버가 떠 있으면 `/api/ocr-arc`로 이미지를 POST하면 같은 결과를 받습니다.
프론트는 신분증 이미지를 이 엔드포인트로 보내면 됩니다.

```
POST /api/ocr-arc  (multipart/form-data, file=이미지)
-> {"status":"success", "data":{name, arc_number, visa_type, nationality}}
```